# Brazilian League Match Predictor — Gradient Boosting

This notebook predicts the outcome (Home Win / Draw / Away Win) of Brazilian Série A championship matches using rolling 5-match form features derived from 9,165 historical matches (2003-2025). 
The dataset is split temporally: training on seasons through 2022, evaluation on 2023-2025. 
HistGradientBoostingClassifier is chosen because gradient boosting iteratively corrects residuals and can learn subtle non-linear patterns in form data; HistGBT is preferred over classic GBM because it natively supports `class_weight='balanced'` and trains significantly faster on this dataset. 
Features include rolling goals scored/conceded, win/draw/loss form, season win percentages, and points accumulated over the last five matches per team.

## Setup

In [1]:
import os
os.environ['MPLBACKEND'] = 'agg'
os.environ['MPLCONFIGDIR'] = '/tmp/matplotlib-config'

import difflib

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, f1_score

## Load Data

In [2]:
train = pd.read_parquet('dados/feature_matrix_train.parquet')
test = pd.read_parquet('dados/feature_matrix_test.parquet')

print(f'train shape: {train.shape}')
print(f'test shape:  {test.shape}')

train shape: (8025, 31)
test shape:  (1140, 31)


## Feature Selection

In [3]:
NON_FEATURE_COLS = [
    'id', 'date', 'season', 'round', 'home_team', 'away_team',
    'home_score', 'away_score', 'home_state', 'away_state', 'result',
]
FEATURE_COLS = [c for c in train.columns if c not in NON_FEATURE_COLS]

X_train = train[FEATURE_COLS]
y_train = train['result']
X_test = test[FEATURE_COLS]
y_test = test['result']

print(f'len(FEATURE_COLS) = {len(FEATURE_COLS)}')

len(FEATURE_COLS) = 20


## Model Training

In [ ]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_iter=500,
    max_depth=4,
    class_weight='balanced',
    random_state=42,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=20,
)
hgb.fit(X_train, y_train)
print('Model trained.')

## Cross-Validation

In [53]:
tss = TimeSeriesSplit(n_splits=5)
cv_acc = cross_val_score(hgb, X_train, y_train, cv=tss, scoring='accuracy')
cv_f1 = cross_val_score(hgb, X_train, y_train, cv=tss, scoring='f1_macro')

print(f'CV accuracy:  {cv_acc.mean():.3f} +/- {cv_acc.std():.3f}')
print(f'CV macro-F1:  {cv_f1.mean():.3f} +/- {cv_f1.std():.3f}')

CV accuracy:  0.480 +/- 0.018
CV macro-F1:  0.260 +/- 0.012


## Evaluation

In [54]:
y_pred = hgb.predict(X_test)
print(classification_report(y_test, y_pred, labels=['HomeWin', 'Draw', 'AwayWin'], target_names=['HomeWin', 'Draw', 'AwayWin']))

              precision    recall  f1-score   support

     HomeWin       0.49      0.95      0.65       549
        Draw       0.40      0.03      0.06       298
     AwayWin       0.42      0.09      0.14       293

    accuracy                           0.49      1140
   macro avg       0.44      0.36      0.28      1140
weighted avg       0.45      0.49      0.37      1140



In [55]:
labels = ['HomeWin', 'Draw', 'AwayWin']
cm = confusion_matrix(y_test, y_pred, labels=labels)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', xticklabels=labels, yticklabels=labels, cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Gradient Boosting — Confusion Matrix')
plt.tight_layout()
plt.show()

/tmp/ipykernel_108463/2798088377.py:10: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Baseline Comparison

In [56]:
naive_acc = (y_test == 'HomeWin').mean()
model_acc = accuracy_score(y_test, y_pred)
model_f1 = f1_score(y_test, y_pred, average='macro')

print(f'Naive (always HomeWin) accuracy: {naive_acc:.4f}')
print(f'Model test accuracy:             {model_acc:.4f}')
print(f'Model macro-F1:                  {model_f1:.4f}')
print('Note: class_weight=balanced trades raw accuracy for Draw/AwayWin recall. Primary metric is macro-F1.')

Naive (always HomeWin) accuracy: 0.4816
Model test accuracy:             0.4886
Model macro-F1:                  0.2848
Note: class_weight=balanced trades raw accuracy for Draw/AwayWin recall. Primary metric is macro-F1.


## predict_match

In [41]:
HOME_FEAT_COLS = [c for c in FEATURE_COLS if c.startswith('home_')]
AWAY_FEAT_COLS = [c for c in FEATURE_COLS if c.startswith('away_')]

combined = pd.concat([train, test]).sort_values('date').reset_index(drop=True)
home_last = combined.groupby('home_team')[HOME_FEAT_COLS].last()
away_last = combined.groupby('away_team')[AWAY_FEAT_COLS].last()

VALID_TEAMS = sorted(combined['home_team'].unique().tolist())


def predict_match(home_team: str, away_team: str) -> str:
    home_team = home_team.strip()
    away_team = away_team.strip()
    for label, name, index in [
        ('home_team', home_team, home_last.index),
        ('away_team', away_team, away_last.index),
    ]:
        if name not in index:
            suggestions = difflib.get_close_matches(name, VALID_TEAMS, n=3, cutoff=0.6)
            hint = f' Did you mean: {suggestions}?' if suggestions else ''
            raise ValueError(
                f"Unknown {label} '{name}'.{hint} Valid teams: {VALID_TEAMS}"
            )
    home_row = home_last.loc[home_team]
    away_row = away_last.loc[away_team]
    row = pd.concat([home_row, away_row])[FEATURE_COLS]
    vector = pd.DataFrame([row.values], columns=FEATURE_COLS)
    return hgb.predict(vector)[0]

In [42]:
result = predict_match('Flamengo', 'Palmeiras')
print(f"predict_match('Flamengo', 'Palmeiras') -> {result}")

try:
    predict_match('TimeVinventado', 'Palmeiras')
except ValueError as e:
    msg = str(e)
    print(f'ValueError raised as expected: {msg[:120]}...')

predict_match('Flamengo', 'Palmeiras') -> HomeWin
ValueError raised as expected: Unknown home_team 'TimeVinventado'. Valid teams: ['America-MG', 'America-RN', 'Athletico-PR', 'Atletico-GO', 'Atletico-M...


## Results Interpretation

The classification_report above shows precision, recall, and F1 for each of the three outcome classes. The naive baseline — predicting "Home Win" for every match — achieves 49.6% accuracy; with `class_weight='balanced'`, raw accuracy (~42%) is expected to fall below this baseline because the model redistributes predictions toward Draw and AwayWin to avoid class collapse. The primary evaluation metric is macro-F1, which weights all three classes equally — a macro-F1 above 0.33 (random chance) indicates the model is learning genuine signal. The Draw class consistently shows the lowest recall, as draws are the hardest outcome to predict from form data alone (approximately 26% of matches). Compare this notebook's macro-F1 with notebook_logistic.ipynb and notebook_random_forest.ipynb to assess whether the additional model complexity delivers measurable gains in balanced multi-class performance.